# Notebook 01 — Fundamentos Matemáticos para LLMs

**Fase 1 · Fundamentos** | [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astorress/llm-lab/blob/main/notebooks/01_fundamentos_matematicos.ipynb)

## Objetivos de aprendizaje

Al completar este notebook serás capaz de:

1. Narrar la historia comprimida de los LLMs y situar cada hito en su contexto.
2. Demostrar el problema del desvanecimiento de gradientes en RNNs con código y gráficas.
3. Manipular tensores de forma fluida con NumPy y PyTorch (reshape, broadcasting, einsum).
4. Entender álgebra lineal como *geometría*: proyecciones, SVD, PCA visual.
5. Implementar backpropagación manual y verificarla con gradient-check numérico.
6. Calcular softmax, cross-entropy y temperatura desde cero.
7. Comparar optimizadores sobre la función de Rosenbrock con trayectorias superpuestas.
8. Implementar greedy, temperatura, top-k y top-p sampling.
9. Calcular entropía y bits por carácter de texto real vs. aleatorio.
10. Interpretar scaling laws (Kaplan y Chinchilla) en gráficas log-log.
11. Derivar la complejidad $O(T^2 d)$ de attention y el crossover con $O(T d^2)$ FFN.

**Prerrequisitos:** cálculo multivariado, álgebra lineal básica, Python, NumPy.

**Tiempo estimado:** ~3 h (SMALL) · ~4.5 h (LARGE, con experimentos de escala)


## Tabla de contenidos

1. [Historia comprimida de los LLMs](#sec1)
2. [Problema de RNNs y vanishing gradients](#sec2)
3. [Operaciones con tensores](#sec3)
4. [Álgebra lineal como geometría](#sec4)
5. [Cálculo: derivadas y regla de la cadena](#sec5)
6. [Gradientes y superficies de error](#sec6)
7. [Backpropagación manual](#sec7)
8. [Softmax, temperatura y cross-entropy](#sec8)
9. [Embeddings y representaciones](#sec9)
10. [Optimizadores](#sec10)
11. [Muestreo (sampling)](#sec11)
12. [Entropía e información](#sec12)
13. [Scaling laws](#sec13)
14. [Complejidad computacional del Transformer](#sec14)
15. [Resumen, ejercicios y referencias](#sec15)


In [ ]:
# ── Instalación de dependencias (silenciosa) ──────────────────────────────
import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

try:
    import plotly
except ImportError:
    _pip("plotly")

try:
    import torch
except ImportError:
    pass  # En Colab ya viene instalado

# ── Detección de hardware ─────────────────────────────────────────────────
import torch, numpy as np, random, os, math, warnings

warnings.filterwarnings("ignore")

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Entorno: {'Google Colab' if IN_COLAB else 'Local'} | Dispositivo: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── Seeds reproducibles ───────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

# ── Estilo de gráficas ────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

matplotlib.rcParams.update({
    "figure.dpi": 110,
    "figure.figsize": (9, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
sns.set_palette("tab10")
print("Setup completo. SEED =", SEED)


In [ ]:
# ── Configuración SMALL / LARGE ───────────────────────────────────────────
PROFILE = "auto"  # "small" | "large" | "auto"

CONFIG = {
    "small": dict(
        d_model=256, n_layers=4, n_heads=4,
        context_len=256, vocab_size=8_000,
        batch_size=32, dtype="fp16",
        n_demo_points=200,
    ),
    "large": dict(
        d_model=768, n_layers=12, n_heads=12,
        context_len=1024, vocab_size=32_000,
        batch_size=64, dtype="bf16",
        n_demo_points=1000,
    ),
}

if PROFILE == "auto":
    if DEVICE == "cuda":
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        PROFILE = "large" if vram_gb >= 14 else "small"
    else:
        PROFILE = "small"

CFG = CONFIG[PROFILE]
print(f"Perfil activo: {PROFILE.upper()}")
print("  " + ", ".join(f"{k}={v}" for k, v in CFG.items()))


---
<a id="sec1"></a>
## 1. Historia comprimida de los LLMs

### 1.1 De n-gramas al Transformer

Los modelos de lenguaje tienen una historia de ~70 años. El árbol genealógico simplificado:

| Época | Paradigma | Representantes |
|-------|-----------|----------------|
| 1950s–1980s | **N-gramas** y probabilidad discreta | Shannon, Jelinek |
| 1986–2010 | **RNNs / LSTMs** | Elman, Hochreiter & Schmidhuber (LSTM 1997) |
| 2013–2017 | **Word embeddings** | Word2Vec, GloVe, FastText |
| 2017 | **Transformer** | Vaswani et al. "Attention is All You Need" |
| 2018–2019 | **Pre-entrenamiento masivo** | BERT (encoders), GPT-1/2 (decoders) |
| 2020–2021 | **Escala** | GPT-3 (175B), Scaling Laws (Kaplan 2020) |
| 2022–2023 | **Alineación + eficiencia** | RLHF, InstructGPT, Chinchilla, LLaMA |
| 2024–2025 | **Multimodal + razonamiento** | GPT-4o, Gemini, Claude 3, LLaMA 3 |

### 1.2 Escala de modelos representativos

| Modelo | Año | Parámetros | Tokens de entrenamiento | d_model | Capas |
|--------|-----|-----------|-------------------------|---------|-------|
| GPT-2 | 2019 | 1.5 B | 40 B | 1600 | 48 |
| GPT-3 | 2020 | 175 B | 300 B | 12288 | 96 |
| Chinchilla | 2022 | 70 B | 1.4 T | — | — |
| LLaMA-2 | 2023 | 70 B | 2 T | 8192 | 80 |
| LLaMA-3 | 2024 | 70 B | 15 T | 8192 | 80 |

**Intuición clave:** más parámetros *no* siempre gana — Chinchilla mostró que el óptimo
de cómputo requiere equilibrar parámetros y tokens. Esto lo veremos formalmente en §13.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Hitos históricos: (año, etiqueta corta, color_grupo)
hitos = [
    (1950, "Shannon\nN-gramas",    "#78909C"),
    (1986, "RNN\nEelman",          "#78909C"),
    (1997, "LSTM\nHochreiter",     "#5C6BC0"),
    (2013, "Word2Vec\nMikolov",    "#5C6BC0"),
    (2017, "Transformer\nVaswani", "#E53935"),
    (2018, "BERT\nDevlin",         "#E53935"),
    (2019, "GPT-2\nOAI",           "#FB8C00"),
    (2020, "GPT-3\n175B",          "#FB8C00"),
    (2020, "Scaling\nLaws Kaplan", "#FB8C00"),
    (2022, "Chinchilla\nDeepMind", "#43A047"),
    (2022, "ChatGPT\nRLHF",        "#43A047"),
    (2023, "LLaMA-2\nMeta",        "#00ACC1"),
    (2024, "LLaMA-3\nMeta",        "#00ACC1"),
    (2024, "GPT-4o\nOAI",          "#7B1FA2"),
]

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.axhline(0, color="#BDBDBD", linewidth=1.5, zorder=0)

# Alternar etiquetas arriba/abajo
for i, (yr, label, color) in enumerate(hitos):
    y = 0.55 if i % 2 == 0 else -0.55
    ax.annotate(label, xy=(yr, 0), xytext=(yr, y),
                ha="center", va="bottom" if y > 0 else "top",
                fontsize=7.5, color=color,
                arrowprops=dict(arrowstyle="-", color=color, lw=1.2))
    ax.plot(yr, 0, "o", color=color, ms=7, zorder=3)

ax.set_xlim(1945, 2026)
ax.set_ylim(-1.2, 1.2)
ax.set_yticks([])
ax.set_xlabel("Año")
ax.set_title("Timeline de hitos en modelado de lenguaje", fontsize=12, fontweight="bold")

# Leyenda de grupos
patches = [
    mpatches.Patch(color="#78909C", label="Era estadística"),
    mpatches.Patch(color="#5C6BC0", label="Deep learning"),
    mpatches.Patch(color="#E53935", label="Transformer"),
    mpatches.Patch(color="#FB8C00", label="Escala GPT"),
    mpatches.Patch(color="#43A047", label="Compute-optimal"),
    mpatches.Patch(color="#00ACC1", label="Open-weights"),
    mpatches.Patch(color="#7B1FA2", label="Multimodal"),
]
ax.legend(handles=patches, ncol=4, fontsize=7.5, loc="upper left", framealpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
# Tabla de parámetros/datos por modelo — verificación numérica
import pandas as pd

data = {
    "Modelo":      ["GPT-2 (L)", "GPT-3",   "Chinchilla", "LLaMA-2-70B", "LLaMA-3-70B"],
    "Params (B)":  [1.5,         175,        70,            70,             70],
    "Tokens (T)":  [0.04,        0.3,        1.4,           2.0,            15.0],
    "d_model":     [1600,        12288,      None,           8192,           8192],
    "Capas":       [48,          96,         None,           80,             80],
    "P/T ratio":   [None, None, None, None, None],  # se calcula abajo
}
df = pd.DataFrame(data)
df["P/T ratio"] = (df["Params (B)"] * 1e9) / (df["Tokens (T)"] * 1e12)
df["P/T ratio"] = df["P/T ratio"].map(lambda x: f"{x:.4f}" if x else "-")
print(df.to_string(index=False))
print("\nNota: Chinchilla propone P/T ≈ 20 tokens/param — LLaMA-3 supera ese ratio.")


---
<a id="sec2"></a>
## 2. Problema de RNNs y vanishing gradients

### 2.1 La RNN simple

Una RNN estándar actualiza su estado oculto con:

$$h_t = \tanh(W_{hh}\,h_{t-1} + W_{xh}\,x_t + b_h)$$

Para calcular $\partial \mathcal{L}/\partial h_0$ necesitamos la regla de la cadena a través
de $T$ pasos:

$$\frac{\partial h_T}{\partial h_0} = \prod_{t=1}^{T} \text{diag}(\tanh'(a_t))\,W_{hh}$$

donde $a_t = W_{hh}\,h_{t-1} + W_{xh}\,x_t$.

### 2.2 Vanishing y exploding gradients

Si el radio espectral $\rho(W_{hh}) < 1$ y $|\tanh'| \leq 1$, el producto converge a 0
exponencialmente — gradiente **desvaneciente**.

Si $\rho(W_{hh}) > 1$, el producto diverge — gradiente **explosivo**.

**Intuición:** es como multiplicar 0.9 × 0.9 × ... por 100 pasos → ≈ 2.66×10⁻⁵.
Con sigmoide la saturación es peor: $\sigma' \leq 0.25$.

### 2.3 La solución: LSTM y, finalmente, Attention

LSTM introduce compuertas que permiten copiar gradientes sin multiplicación repetida
(conexiones residuales en el tiempo). El Transformer va más lejos: conecta *directamente*
cada par de posiciones mediante self-attention $O(T^2)$ — sin pasos recurrentes.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(SEED)

def gradient_norm_rnn(T_max=150, hidden=32, activation="tanh"):
    """Simula la norma del gradiente backprop'd a través de T pasos."""
    W = np.random.randn(hidden, hidden) * 0.5  # radio espectral < 1
    norms = []
    grad = np.ones(hidden)  # gradiente inicial (norma 1)
    for t in range(T_max):
        h = np.random.randn(hidden)
        if activation == "tanh":
            dphi = 1 - np.tanh(h) ** 2          # deriv. tanh, máx 1
        else:  # sigmoide
            sig = 1 / (1 + np.exp(-h))
            dphi = sig * (1 - sig)               # deriv. sigmoide, máx 0.25
        J = dphi[:, None] * W                    # Jacobiano local  (H, H)
        grad = J.T @ grad
        norms.append(np.linalg.norm(grad))
    return np.array(norms)

T = 150
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, act, title in zip(axes, ["tanh", "sigmoid"], ["tanh", "sigmoide"]):
    norms = gradient_norm_rnn(T_max=T, activation=act)
    ax.semilogy(range(1, T+1), norms, lw=1.8, color="#E53935" if act=="sigmoid" else "#1565C0")
    ax.set_xlabel("Longitud de secuencia $T$")
    ax.set_ylabel("Norma del gradiente $\|\partial h_T / \partial h_0\|$")
    ax.set_title(f"Vanishing gradient — activación {title}", fontweight="bold")
    ax.axhline(1e-6, color="gray", ls="--", lw=1, label="umbral 1e-6")
    ax.legend(fontsize=9)

plt.suptitle("El gradiente se desvanece exponencialmente en RNNs", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

# Verificación numérica
t50_tanh = gradient_norm_rnn(T_max=50)[-1]
print(f"Norma del gradiente en T=50 (tanh): {t50_tanh:.2e}")
print("¿Menor que 1e-4?", t50_tanh < 1e-4)


### 2.4 LSTM: la solución recurrente

La LSTM introduce un **cell state** $c_t$ que fluye sin multiplicación:

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$

Las compuertas $f_t$ (forget), $i_t$ (input), $o_t$ (output) son sigmas ∈ (0,1).
El gradiente puede fluir por $c$ sin multiplicarse por $W_{hh}$ repetidamente,
siempre que $f_t \approx 1$.

> **Analogía:** es como una autopista paralela al paso recurrente normal, donde
> el gradiente puede viajar sin peajes multiplicativos.


---
<a id="sec3"></a>
## 3. Operaciones con tensores

Los tensores son la moneda de cambio de los LLMs. La convención **B×T×C** (Karpathy):

- **B** — batch size: cuántas secuencias procesamos en paralelo
- **T** — time / longitud: número de tokens en la secuencia
- **C** — channels / d_model: dimensión del embedding

### 3.1 Operaciones esenciales

| Operación | NumPy | PyTorch |
|-----------|-------|---------|
| Reshape | `a.reshape(B,T,C)` | `x.view(B,T,C)` |
| Transponer | `a.T` | `x.transpose(-2,-1)` |
| Producto matricial | `A @ B` | `torch.matmul(A,B)` |
| Einstein sum | `np.einsum("btc,btd->bcd", q, k)` | `torch.einsum(...)` |
| Broadcasting | automático | automático |


In [ ]:
import torch, numpy as np

B, T, C = 2, 8, 16  # batch, tiempo, canales
torch.manual_seed(SEED)

# Crear y reshape
x = torch.randn(B, T, C)           # (B, T, C)
x_flat = x.view(B, T*C)            # (B, T*C)
x_back = x_flat.view(B, T, C)      # (B, T, C)
assert torch.allclose(x, x_back), "reshape → reshape debe ser identidad"

# Transponer los últimos 2 ejes (útil en attention)
x_t = x.transpose(-2, -1)          # (B, C, T)
print(f"x:     {tuple(x.shape)}")
print(f"x_t:   {tuple(x_t.shape)}")

# Producto de matrices QK^T (atención cruda)
Q = torch.randn(B, T, C)           # (B, T, C)
K = torch.randn(B, T, C)           # (B, T, C)
scores_mm   = Q @ K.transpose(-2, -1)              # (B, T, T)
scores_ein  = torch.einsum("btc,bsc->bts", Q, K)  # (B, T, T)
assert torch.allclose(scores_mm, scores_ein), "matmul == einsum"
print(f"scores: {tuple(scores_mm.shape)}  ✓")

# Broadcasting: suma un bias (C,) sobre un tensor (B,T,C)
bias = torch.zeros(C)              # (C,)
out = x + bias                     # broadcasting automático → (B, T, C)
print(f"broadcasting: x+bias → {tuple(out.shape)}")


In [ ]:
# Ejemplos de einsum para operaciones frecuentes en Transformers
import torch
torch.manual_seed(SEED)

B, T, H, D = 2, 8, 4, 16  # batch, tiempo, heads, dim por head

# Caso 1: attention scores multi-head
Q = torch.randn(B, H, T, D)   # (B, H, T, D)
K = torch.randn(B, H, T, D)   # (B, H, T, D)
scores = torch.einsum("bhid,bhjd->bhij", Q, K)  # (B, H, T, T)
print("Multi-head scores:", tuple(scores.shape))

# Caso 2: weighted sum de values
V = torch.randn(B, H, T, D)   # (B, H, T, D)
attn = torch.softmax(scores / D**0.5, dim=-1)  # (B, H, T, T)
out = torch.einsum("bhij,bhjd->bhid", attn, V) # (B, H, T, D)
print("Attention output:", tuple(out.shape))

# Caso 3: outer product (usado en multiplicación de embeddings)
a = torch.randn(4)
b = torch.randn(5)
outer = torch.einsum("i,j->ij", a, b)  # (4, 5)
print("Outer product:", tuple(outer.shape))


---
<a id="sec4"></a>
## 4. Álgebra lineal como geometría

Un LLM *es* álgebra lineal: cada proyección $W \in \mathbb{R}^{d \times d}$ rota,
escala y refleja el espacio de representaciones.

### 4.1 Transformaciones 2D y su efecto visual

Dada una matriz $A \in \mathbb{R}^{2 \times 2}$, su efecto sobre una nube de puntos
nos dice si comprime, expande, rota o cizalla el espacio.

### 4.2 SVD — la anatomía de cualquier matriz

Toda matriz $W \in \mathbb{R}^{m \times n}$ tiene descomposición:

$$W = U \Sigma V^T$$

- $U \in \mathbb{R}^{m \times m}$: rotación en el espacio de salida
- $\Sigma \in \mathbb{R}^{m \times n}$: escala en cada dirección singular
- $V \in \mathbb{R}^{n \times n}$: rotación en el espacio de entrada

**Rank-$k$ approximation:** tomando los $k$ valores singulares más grandes,
$W_k = U_k \Sigma_k V_k^T$, reconstruimos la mayor varianza posible con solo $k$ parámetros.
Esto es la base de **LoRA** y la compresión de pesos.

### 4.3 PCA como SVD centrado

PCA = SVD sobre la matriz de datos centrada. Las componentes principales son
exactamente los vectores singulares de $X - \bar{X}$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(SEED)

# Nube de puntos original (círculo unitario)
theta = np.linspace(0, 2*np.pi, 200)
X = np.stack([np.cos(theta), np.sin(theta)])  # (2, 200)

# Matrices de transformación
transforms = {
    "Identidad $I$":          np.eye(2),
    "Escala (2x, 0.5y)":      np.diag([2.0, 0.5]),
    "Rotación 45°":           np.array([[np.cos(np.pi/4), -np.sin(np.pi/4)],
                                         [np.sin(np.pi/4),  np.cos(np.pi/4)]]),
    "Cizalla":                np.array([[1, 0.8], [0, 1]]),
    "Reflexión eje y":         np.array([[-1, 0], [0, 1]]),
    "Proyección + escala":     np.array([[1.5, 0.5], [0.3, 0.9]]),
}

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes = axes.ravel()

for ax, (title, A) in zip(axes, transforms.items()):
    X_t = A @ X  # transformación: (2, 2) @ (2, 200) → (2, 200)
    ax.plot(X[0], X[1], "o", ms=2, alpha=0.4, color="#90A4AE", label="original")
    ax.plot(X_t[0], X_t[1], "o", ms=2.5, alpha=0.8, color="#E53935", label="transformado")
    # Vectores columna de A (base transformada)
    for col, color in zip(A.T, ["#1565C0", "#2E7D32"]):
        ax.annotate("", xy=col, xytext=(0,0),
                    arrowprops=dict(arrowstyle="->", color=color, lw=2))
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=9)
    ax.axhline(0, color="#BDBDBD", lw=0.7); ax.axvline(0, color="#BDBDBD", lw=0.7)
    # valor det
    ax.text(0.02, 0.97, f"det={np.linalg.det(A):.2f}", transform=ax.transAxes,
            fontsize=8, va="top", color="#333")

axes[0].legend(fontsize=8)
plt.suptitle("Efecto de transformaciones lineales sobre una nube de puntos", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(SEED)

# Matriz de ejemplo: embedding de palabras sintético (20 palabras x 8 dims)
W = np.random.randn(20, 8) * 2

U, s, Vt = np.linalg.svd(W, full_matrices=False)
# U: (20, 8), s: (8,), Vt: (8, 8)

# Reconstrucción rank-k y error de Frobenius
ks = list(range(1, 9))
errors = []
for k in ks:
    W_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    errors.append(np.linalg.norm(W - W_k, "fro"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfica 1: valores singulares (espectro)
axes[0].bar(range(1, len(s)+1), s, color="#1565C0", alpha=0.8)
axes[0].set_xlabel("Componente singular")
axes[0].set_ylabel("Valor singular $\sigma_i$")
axes[0].set_title("Espectro SVD de la matriz $W$")
variance_explained = np.cumsum(s**2) / np.sum(s**2)
ax2 = axes[0].twinx()
ax2.plot(range(1, len(s)+1), variance_explained * 100, "s--", color="#E53935",
         ms=6, label="% varianza acumulada")
ax2.set_ylabel("Varianza explicada (%)", color="#E53935")
ax2.legend(fontsize=9)

# Gráfica 2: error Frobenius vs rank
axes[1].plot(ks, errors, "o-", color="#43A047", lw=2, ms=7)
axes[1].set_xlabel("Rank $k$ de la aproximación")
axes[1].set_ylabel("Error Frobenius $\|W - W_k\|_F$")
axes[1].set_title("Compresión rank-$k$: error vs. rank\n(base de LoRA)")
for k, e in zip(ks, errors):
    axes[1].text(k, e + 0.05, f"{e:.2f}", ha="center", fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nRank-1 captura {variance_explained[0]*100:.1f}% de la varianza")
print(f"Rank-3 captura {variance_explained[2]*100:.1f}% de la varianza")
print(f"Rank-8 (full): error = {errors[-1]:.2e} (≈ 0)")


---
<a id="sec5"></a>
## 5. Cálculo: derivadas y regla de la cadena

### 5.1 Derivada como razón de cambio local

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

En redes neuronales usamos la **aproximación numérica** para verificar gradientes:

$$\frac{\partial L}{\partial \theta_i} \approx \frac{L(\theta + \epsilon e_i) - L(\theta - \epsilon e_i)}{2\epsilon}$$

### 5.2 Regla de la cadena vectorial

Para $L = f(g(\mathbf{x}))$:

$$\frac{\partial L}{\partial x_j} = \sum_k \frac{\partial L}{\partial f_k} \cdot \frac{\partial f_k}{\partial g} \cdot \frac{\partial g}{\partial x_j}$$

En notación matricial: $\nabla_x L = J_g^T \nabla_g L$ donde $J_g$ es el Jacobiano de $g$.

### 5.3 Activaciones clave y sus derivadas

| Función | $f(x)$ | $f'(x)$ | Problema |
|---------|--------|---------|---------|
| Sigmoide | $\frac{1}{1+e^{-x}}$ | $f(1-f)$ | Saturación: máx 0.25 |
| Tanh | $\frac{e^x-e^{-x}}{e^x+e^{-x}}$ | $1-f^2$ | Saturación: máx 1 |
| ReLU | $\max(0,x)$ | $\mathbf{1}[x>0]$ | Neuronas muertas |
| GELU | $x\,\Phi(x)$ | $\Phi(x) + x\,\phi(x)$ | Usado en GPT |
| SiLU/Swish | $x\,\sigma(x)$ | $\sigma(x)(1+x(1-\sigma(x)))$ | Usado en LLaMA |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erf  # para GELU

x = np.linspace(-4, 4, 500)

def sigmoid(x): return 1 / (1 + np.exp(-x))
def d_sigmoid(x): s = sigmoid(x); return s * (1 - s)
def d_tanh(x): return 1 - np.tanh(x)**2
def relu(x): return np.maximum(0, x)
def d_relu(x): return (x > 0).astype(float)
def gelu(x): return x * 0.5 * (1 + erf(x / np.sqrt(2)))
def d_gelu(x):
    from scipy.stats import norm
    return 0.5 * (1 + erf(x/np.sqrt(2))) + x * norm.pdf(x)
def silu(x): return x * sigmoid(x)
def d_silu(x): s = sigmoid(x); return s * (1 + x * (1 - s))

acts = {
    "Sigmoide": (sigmoid, d_sigmoid, "#E53935"),
    "Tanh":     (np.tanh, d_tanh,    "#1565C0"),
    "ReLU":     (relu,    d_relu,    "#43A047"),
    "GELU":     (gelu,    d_gelu,    "#FB8C00"),
    "SiLU":     (silu,    d_silu,    "#7B1FA2"),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for name, (f, df, color) in acts.items():
    y  = f(x)
    dy = df(x)
    axes[0].plot(x, y,  lw=2, color=color, label=name)
    axes[1].plot(x, dy, lw=2, color=color, label=name)

for ax, title in zip(axes, ["Activaciones $f(x)$", "Derivadas $f'(x)$"]):
    ax.set_xlim(-4, 4)
    ax.axhline(0, color="#BDBDBD", lw=0.8)
    ax.axvline(0, color="#BDBDBD", lw=0.8)
    ax.set_xlabel("$x$"); ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=9)

# Anotación: saturación de sigmoide
axes[1].annotate("máx(σ')=0.25", xy=(0, 0.25), xytext=(1.5, 0.35),
                 arrowprops=dict(arrowstyle="->", color="#E53935"),
                 color="#E53935", fontsize=9)
plt.tight_layout()
plt.show()


---
<a id="sec6"></a>
## 6. Gradientes y superficies de error

El **gradiente** $\nabla_\theta L$ es el vector que apunta en la dirección de máximo
aumento de $L$. Descenso de gradiente actualiza:

$$\theta \leftarrow \theta - \eta \, \nabla_\theta L(\theta)$$

### 6.1 Intuición geométrica

- La superficie de pérdida en $\mathbb{R}^2$ es un relieve topográfico.
- Los valles son mínimos (deseados), los picos son máximos, las sillas son puntos críticos.
- El gradiente en cada punto es perpendicular a las curvas de nivel.

### 6.2 Función de Rosenbrock

$$f(x, y) = (1-x)^2 + 100(y - x^2)^2$$

Mínimo global en $(1, 1)$, pero con un valle estrecho y curvado que desafía a GD simple.
Usaremos esta función en §10 para comparar optimizadores.


In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

# Superficie de Rosenbrock
x = np.linspace(-2, 2, 100)
y = np.linspace(-1, 3, 100)
X, Y = np.meshgrid(x, y)
Z = (1 - X)**2 + 100 * (Y - X**2)**2
Z_log = np.log1p(Z)  # log-scale para visualizar mejor

# Campo de gradientes (submuestreado)
xi = np.linspace(-2, 2, 14)
yi = np.linspace(-1, 3, 14)
Xi, Yi = np.meshgrid(xi, yi)
dfdx = -2*(1 - Xi) - 400*Xi*(Yi - Xi**2)
dfdy = 200*(Yi - Xi**2)
norm = np.sqrt(dfdx**2 + dfdy**2) + 1e-8
dfdx_n = dfdx / norm * 0.15  # normalizar para visualización
dfdy_n = dfdy / norm * 0.15

fig = go.Figure()

# Superficie 3D
fig.add_trace(go.Surface(
    x=X, y=Y, z=Z_log,
    colorscale="Viridis", opacity=0.85,
    colorbar=dict(title="log(1+f)", len=0.6),
    showscale=True,
))

# Mínimo global
fig.add_trace(go.Scatter3d(
    x=[1], y=[1], z=[0],
    mode="markers+text",
    marker=dict(size=8, color="red"),
    text=["mínimo (1,1)"], textposition="top center",
    name="Mínimo",
))

fig.update_layout(
    title="Función de Rosenbrock — superficie log-escalada",
    scene=dict(
        xaxis_title="x", yaxis_title="y", zaxis_title="log(1+f)",
        camera=dict(eye=dict(x=1.7, y=-1.7, z=1.2)),
    ),
    width=750, height=500,
)
fig.show()
print("Mínimo global de Rosenbrock: f(1,1) =", (1-1)**2 + 100*(1 - 1**2)**2)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Superficie 2D con curvas de nivel + campo de gradientes
x = np.linspace(-2, 2, 300)
y = np.linspace(-1, 3, 300)
X, Y = np.meshgrid(x, y)
Z = (1 - X)**2 + 100 * (Y - X**2)**2

fig, ax = plt.subplots(figsize=(7, 6))
levels = np.logspace(0, 3.5, 18)
cs = ax.contourf(X, Y, Z, levels=levels, cmap="viridis_r", alpha=0.7)
ax.contour(X, Y, Z, levels=levels, colors="white", linewidths=0.5, alpha=0.5)
plt.colorbar(cs, ax=ax, label="$f(x,y)$")

# Campo de gradientes
xi = np.linspace(-1.8, 1.8, 12)
yi = np.linspace(-0.8, 2.8, 12)
Xi, Yi = np.meshgrid(xi, yi)
dfdx = -2*(1 - Xi) - 400*Xi*(Yi - Xi**2)
dfdy = 200*(Yi - Xi**2)
norm = np.sqrt(dfdx**2 + dfdy**2) + 1e-8
ax.quiver(Xi, Yi, -dfdx/norm, -dfdy/norm,
          color="white", alpha=0.7, scale=30, width=0.003)

ax.plot(1, 1, "r*", ms=14, label="mínimo (1,1)")
ax.set_xlabel("$x$"); ax.set_ylabel("$y$")
ax.set_title("Rosenbrock: curvas de nivel + campo de gradientes\n(flechas apuntan hacia el descenso)")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()


---
<a id="sec7"></a>
## 7. Backpropagación manual

### 7.1 Derivación completa de una red de 2 capas

Para la red:

$$\mathbf{z}^{(1)} = W^{(1)}\mathbf{x} + \mathbf{b}^{(1)}, \quad
\mathbf{a}^{(1)} = \text{ReLU}(\mathbf{z}^{(1)})$$

$$\mathbf{z}^{(2)} = W^{(2)}\mathbf{a}^{(1)} + \mathbf{b}^{(2)}, \quad
\hat{\mathbf{y}} = \text{softmax}(\mathbf{z}^{(2)})$$

La pérdida cross-entropy: $L = -\sum_k y_k \log \hat{y}_k$

**Gradientes (forward → backward):**

$$\frac{\partial L}{\partial \mathbf{z}^{(2)}} = \hat{\mathbf{y}} - \mathbf{y} \quad \text{(derivada softmax+CE combinada)}$$

$$\frac{\partial L}{\partial W^{(2)}} = \frac{\partial L}{\partial \mathbf{z}^{(2)}} \cdot (\mathbf{a}^{(1)})^T$$

$$\frac{\partial L}{\partial \mathbf{a}^{(1)}} = (W^{(2)})^T \cdot \frac{\partial L}{\partial \mathbf{z}^{(2)}}$$

$$\frac{\partial L}{\partial \mathbf{z}^{(1)}} = \frac{\partial L}{\partial \mathbf{a}^{(1)}} \odot \mathbf{1}[\mathbf{z}^{(1)} > 0]$$

$$\frac{\partial L}{\partial W^{(1)}} = \frac{\partial L}{\partial \mathbf{z}^{(1)}} \cdot \mathbf{x}^T$$

### 7.2 Gradient check numérico

Verificamos con diferencias finitas centradas ($\varepsilon = 10^{-5}$):

$$g_i^{\text{num}} = \frac{L(\theta_i + \varepsilon) - L(\theta_i - \varepsilon)}{2\varepsilon}$$

El error relativo debe ser $< 10^{-6}$ para confirmar que el backprop es correcto.


In [ ]:
import numpy as np

np.random.seed(SEED)

class TwoLayerNet:
    """Red de 2 capas: ReLU + Softmax + Cross-Entropy."""

    def __init__(self, d_in, d_hidden, d_out):
        scale = 0.1
        self.W1 = np.random.randn(d_hidden, d_in)   * scale  # (H, D)
        self.b1 = np.zeros(d_hidden)                          # (H,)
        self.W2 = np.random.randn(d_out, d_hidden)  * scale  # (C, H)
        self.b2 = np.zeros(d_out)                             # (C,)

    def forward(self, x, y):
        """x: (D,), y: (C,) one-hot. Retorna (loss, cache)."""
        # Capa 1
        z1 = self.W1 @ x + self.b1                   # (H,)
        a1 = np.maximum(0, z1)                        # (H,) ReLU
        # Capa 2
        z2 = self.W2 @ a1 + self.b2                  # (C,)
        # Softmax numericamente estable
        z2_s = z2 - z2.max()
        exp_z2 = np.exp(z2_s)
        y_hat = exp_z2 / exp_z2.sum()                # (C,)
        # Cross-entropy
        loss = -np.sum(y * np.log(y_hat + 1e-12))
        cache = (x, z1, a1, z2, y_hat)
        return loss, cache

    def backward(self, y, cache):
        """Retorna gradientes de todos los parámetros."""
        x, z1, a1, z2, y_hat = cache
        grads = {}
        # Grad softmax+CE: dL/dz2 = y_hat - y
        dz2 = y_hat - y                              # (C,)
        grads["W2"] = np.outer(dz2, a1)             # (C, H)
        grads["b2"] = dz2                            # (C,)
        # Grad hacia capa 1
        da1 = self.W2.T @ dz2                       # (H,)
        dz1 = da1 * (z1 > 0).astype(float)          # (H,) ReLU
        grads["W1"] = np.outer(dz1, x)              # (H, D)
        grads["b1"] = dz1                            # (H,)
        return grads

    def all_params(self):
        return {"W1": self.W1, "b1": self.b1, "W2": self.W2, "b2": self.b2}

    def set_param(self, name, val):
        setattr(self, name, val)


# ── Inicializar
D, H, C = 10, 8, 5
net = TwoLayerNet(D, H, C)
x = np.random.randn(D)
y = np.zeros(C); y[2] = 1.0  # clase 2

loss, cache = net.forward(x, y)
grads = net.backward(y, cache)
print(f"Forward loss: {loss:.6f}")

# ── Gradient check numérico ───────────────────────────────────────────────
eps = 1e-5
max_rel_err = 0.0
results = []

for pname, p in net.all_params().items():
    p_flat = p.flatten()
    for i in range(len(p_flat)):
        # L(θ + ε)
        p_plus = p_flat.copy(); p_plus[i] += eps
        net.set_param(pname, p_plus.reshape(p.shape))
        lp, _ = net.forward(x, y)
        # L(θ - ε)
        p_minus = p_flat.copy(); p_minus[i] -= eps
        net.set_param(pname, p_minus.reshape(p.shape))
        lm, _ = net.forward(x, y)
        # Restaurar
        net.set_param(pname, p)
        # Gradiente numérico
        g_num = (lp - lm) / (2 * eps)
        g_ana = grads[pname].flatten()[i]
        rel_err = abs(g_num - g_ana) / (abs(g_num) + abs(g_ana) + 1e-10)
        if rel_err > max_rel_err:
            max_rel_err = rel_err

print(f"\nGradient check — error relativo máximo: {max_rel_err:.2e}")
assert max_rel_err < 1e-5, f"Gradient check FALLÓ: {max_rel_err:.2e}"
print("✓ Gradient check PASÓ (error < 1e-5)")


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(SEED)

# Red equivalente en PyTorch
D, H, C = 10, 8, 5
net_torch = nn.Sequential(
    nn.Linear(D, H),
    nn.ReLU(),
    nn.Linear(H, C),
)
# Copiar pesos de la implementación manual
with torch.no_grad():
    net_torch[0].weight.copy_(torch.tensor(net.W1, dtype=torch.float32))
    net_torch[0].bias.copy_(torch.tensor(net.b1, dtype=torch.float32))
    net_torch[2].weight.copy_(torch.tensor(net.W2, dtype=torch.float32))
    net_torch[2].bias.copy_(torch.tensor(net.b2, dtype=torch.float32))

x_t = torch.tensor(net.W1 @ x * 0 + x, dtype=torch.float32)  # mismo x
y_t = torch.tensor(y, dtype=torch.float32)

logits = net_torch(x_t)
loss_t = nn.CrossEntropyLoss()(logits.unsqueeze(0), torch.argmax(y_t).unsqueeze(0))
loss_t.backward()

# Comparar gradientes W1
g_manual = torch.tensor(grads["W1"], dtype=torch.float32)
g_torch  = net_torch[0].weight.grad

match = torch.allclose(g_manual, g_torch, atol=1e-5)
print(f"Pérdida PyTorch:   {loss_t.item():.6f}")
print(f"Pérdida manual:    {loss:.6f}")
print(f"torch.allclose(W1_grad_manual, W1_grad_torch): {match}")
assert match, "Los gradientes no coinciden!"
print("✓ Backprop manual verificado contra PyTorch")


---
<a id="sec8"></a>
## 8. Softmax, temperatura y cross-entropy

### 8.1 Softmax

$$\text{softmax}(\mathbf{z})_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

**Problema de overflow:** si $z_i$ es grande, $e^{z_i}$ se desborda (inf en float32
con $z > 88$). Solución: restar el máximo antes de exponenciar.

$$\text{softmax}(\mathbf{z})_i = \frac{e^{z_i - z_{\max}}}{\sum_j e^{z_j - z_{\max}}}$$

Este truco es matemáticamente idéntico pero numéricamente estable.

### 8.2 Temperatura

Introducimos $T$ para controlar la "nitidez" de la distribución:

$$\text{softmax}(\mathbf{z}/T)_i = \frac{e^{z_i/T}}{\sum_j e^{z_j/T}}$$

- $T \to 0$: distribución *greedy* (masa en el argmax)
- $T = 1$: distribución original
- $T \to \infty$: distribución *uniforme*

### 8.3 Cross-entropy

$$\mathcal{L}_{\text{CE}} = -\sum_i y_i \log \hat{y}_i = -\log \hat{y}_{y^*}$$

donde $y^*$ es la clase correcta. Equivalente a la log-likelihood negativa.


In [ ]:
import numpy as np

# Demostración de overflow sin truco del máximo
z = np.array([1000.0, 1001.0, 1002.0])

# Softmax sin estabilización → overflow
exp_raw = np.exp(z)
print("exp sin estabilizar:", exp_raw)  # [inf inf inf]

# Softmax estable
z_stable = z - z.max()
exp_stable = np.exp(z_stable)
softmax_stable = exp_stable / exp_stable.sum()
print("exp estabilizado:   ", np.exp(z_stable))
print("softmax estable:    ", softmax_stable)

# Verificación: mismo resultado que scipy
from scipy.special import softmax as scipy_softmax
assert np.allclose(softmax_stable, scipy_softmax(z)), "Deben ser iguales"
print("✓ El truco del máximo da el mismo resultado matemático")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(SEED)

# Logits ejemplo (vocabulario pequeño de 8 tokens)
vocab_size = 8
logits = np.array([3.0, 1.5, -0.5, 2.2, 0.0, -1.0, 1.0, 0.8])
token_labels = [f"t{i}" for i in range(vocab_size)]

temps = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
colors = plt.cm.RdYlBu_r(np.linspace(0.1, 0.9, len(temps)))

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.ravel()

for ax, T, color in zip(axes, temps, colors):
    z = logits / T
    z_s = z - z.max()
    probs = np.exp(z_s) / np.exp(z_s).sum()
    bars = ax.bar(token_labels, probs, color=color, edgecolor="white", lw=0.5)
    ax.set_ylim(0, 1)
    ax.set_title(f"Temperatura $T = {T}$", fontweight="bold", fontsize=10)
    ax.set_ylabel("$P(\\text{token})$")
    # Entropía
    H = -np.sum(probs * np.log(probs + 1e-12))
    ax.text(0.97, 0.97, f"H={H:.2f}", transform=ax.transAxes,
            ha="right", va="top", fontsize=9, color="#555")

plt.suptitle("Efecto de la temperatura en softmax\n(T→0: concentrado, T→∞: uniforme)",
             fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np

def cross_entropy(y_true, y_pred):
    """y_true: one-hot, y_pred: probabilidades."""
    return -np.sum(y_true * np.log(y_pred + 1e-12))

# Ejemplo verificable a mano
# Vocabulario: 3 clases. Clase correcta = 1
y_true = np.array([0, 1, 0])
y_pred_good = np.array([0.05, 0.90, 0.05])  # predicción confiada y correcta
y_pred_bad  = np.array([0.60, 0.20, 0.20])  # predicción incorrecta

ce_good = cross_entropy(y_true, y_pred_good)
ce_bad  = cross_entropy(y_true, y_pred_bad)
print(f"CE con buena predicción:  {ce_good:.4f}  (= -log(0.90) = {-np.log(0.90):.4f})")
print(f"CE con mala predicción:   {ce_bad:.4f}   (= -log(0.20) = {-np.log(0.20):.4f})")
print(f"\nPerplexity buena: {np.exp(ce_good):.3f}")
print(f"Perplexity mala:  {np.exp(ce_bad):.3f}")

# Verificar con PyTorch
import torch
import torch.nn.functional as F

logits_good = torch.log(torch.tensor(y_pred_good, dtype=torch.float32))
target = torch.tensor(1)  # índice clase correcta
ce_torch = F.cross_entropy(logits_good.unsqueeze(0), target.unsqueeze(0))
print(f"\nVerificación PyTorch CE: {ce_torch.item():.4f} (manual: {ce_good:.4f})")
assert abs(ce_torch.item() - ce_good) < 1e-4
print("✓ Cross-entropy manual = PyTorch")


---
<a id="sec9"></a>
## 9. Embeddings y representaciones

Un **embedding** mapea un token discreto $v \in \{0,...,|V|-1\}$ a un vector denso
$\mathbf{e}_v \in \mathbb{R}^d$. La capa `nn.Embedding` es simplemente una tabla
de búsqueda (lookup table):

$$E \in \mathbb{R}^{|V| \times d}, \quad \mathbf{e}_v = E[v, :]$$

### 9.1 Propiedades algebraicas

- **Similaridad coseno:** $\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{|\mathbf{a}|\,|\mathbf{b}|}$
- **Analogías:** $\mathbf{e}(\text{rey}) - \mathbf{e}(\text{hombre}) + \mathbf{e}(\text{mujer}) \approx \mathbf{e}(\text{reina})$ (Word2Vec)
- **Distancia L2:** $d(\mathbf{a}, \mathbf{b}) = \|\mathbf{a} - \mathbf{b}\|_2$

### 9.2 Embeddings posicionales

En el Transformer (sin recurrencia), el modelo necesita saber la posición de cada token.
Las **positional encodings sinusoidales** de Vaswani et al.:

$$PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \quad
PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

**Intuición:** cada dimensión oscila a una frecuencia distinta, como las agujas de
un reloj — permiten al modelo distinguir cualquier posición y aprender desplazamientos relativos.


In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt

torch.manual_seed(SEED)

# Embedding lookup table
V, d = 100, 32  # vocab pequeño, dim embedding
emb_layer = torch.nn.Embedding(V, d)
# Tokens de ejemplo
tokens = torch.tensor([5, 12, 23, 5])  # token 5 aparece dos veces
vecs = emb_layer(tokens)               # (4, 32)
print("Token 5 ==  Token 5 (misma posición en tabla)?",
      torch.allclose(vecs[0], vecs[3]))

# Similaridad coseno entre pares
import torch.nn.functional as F
v0_n = F.normalize(vecs[0].unsqueeze(0))  # (1, 32)
v1_n = F.normalize(vecs[1].unsqueeze(0))
cos_sim = (v0_n * v1_n).sum().item()
print(f"Similaridad coseno(token5, token12) = {cos_sim:.4f}  (aleatorio → ~0)")

# Visualizar positional encodings
T, d_model = 64, 64
pe = torch.zeros(T, d_model)
positions = torch.arange(T).unsqueeze(1).float()       # (T, 1)
div_term = torch.exp(torch.arange(0, d_model, 2).float()
                     * (-np.log(10000.0) / d_model))   # (d/2,)
pe[:, 0::2] = torch.sin(positions * div_term)          # dimensiones pares
pe[:, 1::2] = torch.cos(positions * div_term)          # dimensiones impares

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(pe.numpy(), aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_xlabel("Dimensión $i$"); axes[0].set_ylabel("Posición $t$")
axes[0].set_title("Positional Encodings sinusoidales")
plt.colorbar(im, ax=axes[0])

# PE de las primeras 4 dimensiones vs posición
for i in range(0, 8, 2):
    axes[1].plot(pe[:, i].numpy(), lw=1.5, label=f"dim {i}")
axes[1].set_xlabel("Posición"); axes[1].set_ylabel("Valor PE")
axes[1].set_title("PE de distintas dimensiones vs posición")
axes[1].legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


---
<a id="sec10"></a>
## 10. Optimizadores

### 10.1 El zoo de optimizadores

| Optimizador | Actualización | Parámetros extra |
|-------------|--------------|-----------------|
| GD | $\theta \leftarrow \theta - \eta g$ | $\eta$ |
| Momentum | $v \leftarrow \beta v + g$; $\theta \leftarrow \theta - \eta v$ | $\beta$ |
| RMSProp | $s \leftarrow \alpha s + (1-\alpha)g^2$; $\theta \leftarrow \theta - \frac{\eta}{\sqrt{s+\varepsilon}}g$ | $\alpha$ |
| Adam | combina momentum + RMSProp con corrección de sesgo | $\beta_1, \beta_2, \varepsilon$ |
| AdamW | Adam + weight decay desacoplado | $\lambda$ |

### 10.2 Adam: derivación

Adam mantiene estimaciones de primer y segundo momento:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t \quad \text{(momentum)}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \quad \text{(varianza)}$$

Corrección de sesgo (los momentos empiezan en 0):

$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$

Actualización:

$$\theta_t = \theta_{t-1} - \frac{\eta}{\sqrt{\hat{v}_t} + \varepsilon}\,\hat{m}_t$$

### 10.3 AdamW y el problema del weight decay en Adam

En Adam, el decaimiento de pesos estándar $\theta \leftarrow (1-\lambda\eta)\theta - ...$
interactúa con la normalización por $\sqrt{v}$. AdamW lo desacopla:

$$\theta_t = \theta_{t-1}(1 - \eta\lambda) - \frac{\eta}{\sqrt{\hat{v}_t}+\varepsilon}\hat{m}_t$$

Esto es lo que usan LLaMA, GPT-4, etc.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(SEED)

def rosenbrock(x, y):
    return (1 - x)**2 + 100*(y - x**2)**2

def grad_rosenbrock(x, y):
    gx = -2*(1 - x) - 400*x*(y - x**2)
    gy = 200*(y - x**2)
    return np.array([gx, gy])

def run_optimizer(opt_name, n_steps=800, lr=1e-3, x0=(-1.5, 0.5)):
    x = np.array(x0, dtype=float)
    path = [x.copy()]
    # Estado interno de cada optimizador
    v = np.zeros(2)      # momentum
    s = np.ones(2)*1e-8  # RMSProp
    m = np.zeros(2)      # Adam m1
    vv = np.zeros(2)     # Adam m2
    beta1, beta2, eps = 0.9, 0.999, 1e-8
    lam = 0.01  # AdamW weight decay

    for t in range(1, n_steps+1):
        g = grad_rosenbrock(*x)
        if opt_name == "GD":
            x -= lr * g
        elif opt_name == "Momentum":
            v = 0.9*v + g
            x -= lr * v
        elif opt_name == "RMSProp":
            s = 0.9*s + 0.1*g**2
            x -= lr * g / (np.sqrt(s) + eps)
        elif opt_name == "Adam":
            m = beta1*m + (1-beta1)*g
            vv = beta2*vv + (1-beta2)*g**2
            mhat = m / (1-beta1**t)
            vhat = vv / (1-beta2**t)
            x -= lr * mhat / (np.sqrt(vhat) + eps)
        elif opt_name == "AdamW":
            m = beta1*m + (1-beta1)*g
            vv = beta2*vv + (1-beta2)*g**2
            mhat = m / (1-beta1**t)
            vhat = vv / (1-beta2**t)
            x = x*(1 - lr*lam) - lr * mhat / (np.sqrt(vhat) + eps)
        path.append(x.copy())
    return np.array(path)

configs = {
    "GD":       (1e-3, "#78909C"),
    "Momentum": (1e-3, "#1565C0"),
    "RMSProp":  (1e-3, "#43A047"),
    "Adam":     (1e-2, "#E53935"),
    "AdamW":    (1e-2, "#FB8C00"),
}

# Fondo: contornos de Rosenbrock
x_arr = np.linspace(-2, 2, 300)
y_arr = np.linspace(-1, 3, 300)
X, Y = np.meshgrid(x_arr, y_arr)
Z = rosenbrock(X, Y)

fig, ax = plt.subplots(figsize=(10, 7))
levels = np.logspace(0, 3.5, 20)
ax.contourf(X, Y, Z, levels=levels, cmap="Greys", alpha=0.55)
ax.contour(X, Y, Z, levels=levels, colors="white", linewidths=0.4, alpha=0.4)

final_losses = {}
for name, (lr, color) in configs.items():
    path = run_optimizer(name, lr=lr)
    ax.plot(path[:, 0], path[:, 1], lw=1.8, alpha=0.85, color=color, label=name)
    ax.plot(path[0, 0], path[0, 1], "o", ms=5, color=color)
    final_losses[name] = rosenbrock(*path[-1])

ax.plot(1, 1, "r*", ms=16, zorder=10, label="mínimo (1,1)")
ax.set_xlim(-2, 2); ax.set_ylim(-1, 3)
ax.set_xlabel("$x$"); ax.set_ylabel("$y$")
ax.set_title("Trayectorias de optimizadores sobre Rosenbrock\n(800 pasos)", fontweight="bold")
ax.legend(fontsize=9, loc="upper right")
plt.tight_layout()
plt.show()

print("\nPérdida final después de 800 pasos:")
for name, loss in final_losses.items():
    print(f"  {name:10s}: {loss:.4f}")


---
<a id="sec11"></a>
## 11. Muestreo (Sampling)

En inferencia, dado un LLM que produce logits $\mathbf{z} \in \mathbb{R}^{|V|}$,
¿cómo seleccionar el siguiente token?

### 11.1 Greedy decoding

$$t^* = \arg\max_i \mathbf{z}_i$$

**Ventaja:** determinista, rápido.
**Problema:** repetitivo y con baja diversidad.

### 11.2 Temperatura

Muestrear de $\text{softmax}(\mathbf{z}/T)$.
$T < 1$ → más concentrado; $T > 1$ → más diverso.

### 11.3 Top-k sampling

Conservar solo los $k$ tokens más probables, redistribuir la masa:

$$P'(t) = \frac{P(t) \cdot \mathbf{1}[t \in \text{Top-}k]}{\sum_{t' \in \text{Top-}k} P(t')}$$

### 11.4 Top-p (nucleus) sampling

Conservar el conjunto mínimo $S$ tal que $\sum_{t \in S} P(t) \geq p$:

$$S_p = \min\{S \subseteq V : \sum_{t \in S} P(t) \geq p\}$$

**Ventaja sobre top-k:** adaptativo — si la distribución es flat, incluye más tokens;
si es concentrada, incluye menos.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(SEED)

# Logits de ejemplo (vocab de 20 tokens)
V = 20
logits = np.array([4.0, 3.5, 2.0, 1.5, 1.0, 0.5, 0.2, -0.5, -1.0, -1.5,
                   -2.0, -2.5, -3.0, -3.2, -3.5, -3.8, -4.0, -4.2, -4.5, -5.0])
token_labels = [f"t{i}" for i in range(V)]

def softmax(z):
    z_s = z - z.max()
    e = np.exp(z_s)
    return e / e.sum()

def top_k_probs(probs, k):
    idx = np.argsort(probs)[::-1]
    mask = np.zeros(V, dtype=bool)
    mask[idx[:k]] = True
    p = probs * mask
    return p / p.sum(), mask

def top_p_probs(probs, p=0.9):
    idx_sorted = np.argsort(probs)[::-1]
    cumsum = np.cumsum(probs[idx_sorted])
    cutoff = np.searchsorted(cumsum, p) + 1
    mask = np.zeros(V, dtype=bool)
    mask[idx_sorted[:cutoff]] = True
    pr = probs * mask
    return pr / pr.sum(), mask, cutoff

# ── Figura comparativa ───────────────────────────────────────────────────
T_val = 1.0
probs_base = softmax(logits / T_val)
probs_k,  mask_k            = top_k_probs(probs_base, k=5)
probs_p,  mask_p, p_cutoff  = top_p_probs(probs_base, p=0.9)
greedy_idx = np.argmax(probs_base)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.ravel()

configs_plot = [
    ("Distribución base (T=1)", probs_base, None, "Base"),
    ("Greedy (argmax)", probs_base, greedy_idx, "Greedy"),
    ("Top-k (k=5)", probs_k, None, "Top-k"),
    ("Top-p / Nucleus (p=0.90)", probs_p, None, "Top-p"),
]

for ax, (title, probs, greedy, lbl) in zip(axes, configs_plot):
    colors = ["#1565C0"] * V
    if lbl == "Greedy":
        colors = ["#E53935" if i == greedy_idx else "#BDBDBD" for i in range(V)]
    elif lbl == "Top-k":
        colors = ["#FB8C00" if mask_k[i] else "#BDBDBD" for i in range(V)]
    elif lbl == "Top-p":
        colors = ["#43A047" if mask_p[i] else "#BDBDBD" for i in range(V)]
    ax.bar(range(V), probs, color=colors, edgecolor="white", lw=0.3)
    ax.set_xticks(range(V))
    ax.set_xticklabels(token_labels, rotation=45, fontsize=7)
    ax.set_ylabel("Probabilidad")
    ax.set_title(title, fontweight="bold", fontsize=9)
    H = -np.sum(probs * np.log(probs + 1e-12))
    ax.text(0.97, 0.97, f"H={H:.2f}", transform=ax.transAxes,
            ha="right", va="top", fontsize=9)

plt.suptitle("Estrategias de muestreo — vocabulario de 20 tokens", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Top-p incluye {p_cutoff} tokens para cubrir el 90% de la masa de probabilidad")


---
<a id="sec12"></a>
## 12. Entropía e información

### 12.1 Entropía de Shannon

$$H(X) = -\sum_i p_i \log_2 p_i \quad \text{[bits]}$$

- **Mínimo:** $H = 0$ si hay certeza ($p_i = 1$ para algún $i$)
- **Máximo:** $H = \log_2 |V|$ si la distribución es uniforme
- **Interpretación:** bits promedio necesarios para codificar un símbolo

### 12.2 Entropía cruzada y perplexidad

$$H(p, q) = -\sum_i p_i \log_2 q_i$$

$$\text{PPL} = 2^{H(p,q)} = e^{\mathcal{L}_{\text{CE}}}$$

Un LLM perfecto que conoce el siguiente token tendría PPL = 1. GPT-3 en inglés tiene PPL ≈ 20.

### 12.3 Bits por carácter (BPC)

$$\text{BPC} = \frac{\text{cross-entropy en nats}}{\ln 2}$$

Texto inglés real: ~1.1–1.3 BPC. Texto aleatorio (uniforme sobre 26 letras): $\log_2 26 \approx 4.7$ BPC.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

def entropy_bits(text):
    """Entropía de caracteres en bits."""
    counts = Counter(text)
    total = len(text)
    probs = np.array([v/total for v in counts.values()])
    return -np.sum(probs * np.log2(probs + 1e-12))

def bpc_uniform(alphabet_size):
    """BPC si la distribución sobre el alfabeto fuera uniforme."""
    return np.log2(alphabet_size)

# Textos de ejemplo (inline, sin descargar nada)
texto_espanol = (
    "El aprendizaje automático es una rama de la inteligencia artificial que "
    "permite a los sistemas aprender y mejorar a partir de la experiencia sin "
    "ser programados explícitamente. Los modelos de lenguaje son el resultado "
    "de décadas de investigación en procesamiento del lenguaje natural."
)

texto_ingles = (
    "Machine learning is a branch of artificial intelligence that enables "
    "systems to learn and improve from experience without being explicitly "
    "programmed. Language models are the result of decades of research in "
    "natural language processing."
)

texto_aleatorio = "".join(np.random.choice(list("abcdefghijklmnopqrstuvwxyz "), 300))

textos = {
    "Español (técnico)": texto_espanol.lower(),
    "Inglés (técnico)": texto_ingles.lower(),
    "Aleatorio (26 letras + espacio)": texto_aleatorio,
}

print("=" * 55)
print(f"{'Texto':<30} {'H (bits)':>10} {'BPC uniforme':>13}")
print("=" * 55)
for nombre, txt in textos.items():
    h = entropy_bits(txt)
    chars_unique = len(set(txt))
    bpc_uni = bpc_uniform(chars_unique)
    print(f"{nombre:<30} {h:>10.3f} {bpc_uni:>13.3f}")
print("=" * 55)
print(f"\nNota: texto real ≈ 4-4.5 bits (menos que uniforme {bpc_uniform(27):.2f} bits)")

# Gráfica: entropía como función del sesgo de la distribución
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribución binaria: p vs H
ps = np.linspace(0.001, 0.999, 500)
H_bin = -(ps * np.log2(ps) + (1-ps) * np.log2(1-ps))
axes[0].plot(ps, H_bin, lw=2, color="#1565C0")
axes[0].set_xlabel("$p$"); axes[0].set_ylabel("$H(p)$ [bits]")
axes[0].set_title("Entropía binaria $H(p, 1-p)$")
axes[0].axvline(0.5, color="#E53935", ls="--", label="máximo H=1 en p=0.5")
axes[0].legend(fontsize=9)

# Distribución categórica con K clases y sesgo
Ks = [2, 5, 10, 26, 100]
alphas = np.linspace(0.1, 5, 200)
for K in Ks:
    Hs = []
    for a in alphas:
        raw = np.exp(-a * np.arange(K))
        p = raw / raw.sum()
        Hs.append(-np.sum(p * np.log2(p + 1e-15)))
    axes[1].plot(alphas, Hs, lw=2, label=f"K={K}")

axes[1].set_xlabel("Concentración α (mayor → más sesgada)")
axes[1].set_ylabel("$H$ [bits]")
axes[1].set_title("Entropía de distribuciones categóricas\nvs concentración")
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()


---
<a id="sec13"></a>
## 13. Scaling Laws

### 13.1 Kaplan et al. (2020) — OpenAI

Los autores observaron que la pérdida de validación sigue relaciones de potencia:

$$L(N) \approx \left(\frac{N_c}{N}\right)^{\alpha_N}, \quad
L(D) \approx \left(\frac{D_c}{D}\right)^{\alpha_D}, \quad
L(C) \approx \left(\frac{C_c}{C}\right)^{\alpha_C}$$

- $N$: número de parámetros; $D$: tokens de entrenamiento; $C$: FLOPs
- Exponentes empíricos: $\alpha_N \approx 0.076$, $\alpha_D \approx 0.095$

**Implicación:** para un presupuesto de cómputo fijo, *escalar parámetros es más eficiente
que escalar datos* (conclusión de Kaplan).

### 13.2 Hoffmann et al. (2022) — Chinchilla

Usando un experimento más controlado (isoFLOP), Chinchilla concluye:

$$N_{\text{opt}} \propto C^{0.5}, \quad D_{\text{opt}} \propto C^{0.5}$$

Es decir, **el número óptimo de tokens es ~20× el número de parámetros**.
Esto revirtió la conclusión de Kaplan: GPT-3 estaba *subentrenado*.

### 13.3 Ley de potencia en log-log

En escala log-log, las leyes de potencia son líneas rectas:

$$\log L = \log L_0 - \alpha \log N$$

La pendiente es $-\alpha$ (negativa: más parámetros → menor pérdida).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parámetros empíricos aproximados de la literatura
alpha_N = 0.076   # Kaplan
alpha_D = 0.095   # Kaplan
N_c     = 8.8e13  # constante de normalización (empírica)
D_c     = 5.4e13

def kaplan_N(N, L_ref=10.0, alpha=alpha_N, N_ref=1e8):
    """Pérdida como función de parámetros."""
    return L_ref * (N_ref / N) ** alpha

def kaplan_D(D, L_ref=10.0, alpha=alpha_D, D_ref=1e9):
    """Pérdida como función de tokens."""
    return L_ref * (D_ref / D) ** alpha

# Rango de parámetros: 10M a 500B
N_arr = np.logspace(7, 11.5, 200)    # 10M → 300B params
D_arr = np.logspace(8, 13, 200)      # 100M → 10T tokens

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# ── Pérdida vs Parámetros ────────────────────────────────────────────────
L_N = kaplan_N(N_arr)
axes[0].loglog(N_arr, L_N, lw=2.5, color="#1565C0", label=f"Kaplan α={alpha_N}")
# Modelos representativos
models_N = {
    "GPT-2(L)": 1.5e9, "GPT-3": 175e9, "Chinchilla": 70e9, "LLaMA-3": 70e9,
}
for mname, mN in models_N.items():
    axes[0].axvline(mN, color="#BDBDBD", ls=":", lw=1)
    axes[0].text(mN, kaplan_N(mN)*1.05, mname, rotation=90, fontsize=7.5, ha="right")
axes[0].set_xlabel("Parámetros $N$"); axes[0].set_ylabel("Pérdida $L$")
axes[0].set_title("Scaling law — parámetros\n(Kaplan 2020)", fontweight="bold")
axes[0].legend(fontsize=9)

# ── Pérdida vs Tokens ────────────────────────────────────────────────────
L_D = kaplan_D(D_arr)
axes[1].loglog(D_arr, L_D, lw=2.5, color="#43A047", label=f"Kaplan α={alpha_D}")
# Chinchilla rule: D_opt = 20 * N → para 70B params
chinchilla_D = 20 * 70e9
axes[1].axvline(chinchilla_D, color="#E53935", ls="--", lw=1.5,
                label="Chinchilla 70B (D=1.4T)")
axes[1].set_xlabel("Tokens de entrenamiento $D$"); axes[1].set_ylabel("Pérdida $L$")
axes[1].set_title("Scaling law — tokens\n(Kaplan / Chinchilla)", fontweight="bold")
axes[1].legend(fontsize=9)

# ── Regla Chinchilla: N_opt vs Presupuesto de cómputo ───────────────────
C_arr = np.logspace(20, 27, 200)   # FLOPs totales
# Approx Chinchilla: N_opt ∝ C^0.5, D_opt ∝ C^0.5
N_chinch = 0.41 * C_arr ** 0.5
D_chinch = C_arr / (6 * N_chinch)
axes[2].loglog(C_arr, N_chinch, lw=2.5, color="#FB8C00", label="$N_{opt}$")
axes[2].loglog(C_arr, D_chinch, lw=2.5, color="#7B1FA2", ls="--", label="$D_{opt}$")
axes[2].set_xlabel("Presupuesto de cómputo $C$ [FLOPs]")
axes[2].set_ylabel("$N_{opt}$ o $D_{opt}$")
axes[2].set_title("Chinchilla: asignación óptima\n$N_{opt} \\propto C^{0.5}$", fontweight="bold")
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

# Verificación numérica: regla 20 tokens/parámetro
N_test = 7e9  # 7B parámetros
D_optimal = 20 * N_test
print(f"Chinchilla: para {N_test/1e9:.0f}B params → D_opt = {D_optimal/1e12:.1f}T tokens")
print(f"LLaMA-2-7B usó 2T tokens → ratio D/N = {2e12/N_test:.0f} (≈ Chinchilla)")


---
<a id="sec14"></a>
## 14. Complejidad computacional del Transformer

### 14.1 Self-Attention: $O(T^2 d)$

Para calcular las matrices $QK^T \in \mathbb{R}^{T \times T}$ para cada par de posiciones:

- $Q, K, V \in \mathbb{R}^{T \times d}$: proyecciones lineales → $O(T d^2)$
- $QK^T$: producto $T \times d$ por $d \times T$ → $O(T^2 d)$
- Softmax + $AV$: → $O(T^2 d)$

**Total self-attention**: $O(T^2 d)$ — cuadrático en la longitud de secuencia.

### 14.2 FFN: $O(T d^2)$

Cada posición pasa por $\text{FFN}(x) = W_2 \, \text{GELU}(W_1 x)$:

- $W_1 \in \mathbb{R}^{4d \times d}$, $W_2 \in \mathbb{R}^{d \times 4d}$
- Por posición: $O(d^2)$; para $T$ posiciones: $O(T d^2)$

### 14.3 Crossover: ¿cuándo domina attention vs FFN?

$$O(T^2 d) \text{ vs } O(T d^2) \Rightarrow \text{crossover en } T = d$$

- Si $T \ll d$: la FFN domina → aumentar $d$ es el cuello de botella
- Si $T \gg d$: la attention domina → longitudes largas son costosas
- Típico: GPT-3 tiene $d=12288$, contexto $T=2048$ → $T < d$, FFN domina

### 14.4 Regla 6ND para estimar FLOPs de entrenamiento

$$C_{\text{train}} \approx 6 N D$$

donde $N$ = parámetros, $D$ = tokens de entrenamiento.
El factor 6 viene de: 2 (forward) × 3 (forward + backward con gradientes de activaciones).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Crossover O(T^2 d) vs O(T d^2) ──────────────────────────────────────
T_arr = np.logspace(1, 5, 500)  # 10 a 100k tokens

d_values = [256, 768, 4096, 12288]  # d_model típicos: mini, GPT-2, LLaMA, GPT-3
colors   = ["#43A047", "#1565C0", "#FB8C00", "#E53935"]
labels   = ["d=256 (SMALL)", "d=768 (GPT-2)", "d=4096 (LLaMA-7B)", "d=12288 (GPT-3)"]

fig, ax = plt.subplots(figsize=(10, 5.5))

for d, color, label in zip(d_values, colors, labels):
    attn_ops = T_arr**2 * d    # O(T^2 d)
    ffn_ops  = T_arr  * d**2   # O(T d^2)
    # Normalizar por el valor en T=d
    norm = d**3
    ax.loglog(T_arr, attn_ops/norm, color=color, lw=2.5, label=f"Attn {label}")
    ax.loglog(T_arr, ffn_ops/norm,  color=color, lw=2.5, ls="--")
    # Marcar crossover en T=d
    ax.axvline(d, color=color, ls=":", alpha=0.5, lw=1.5)
    ax.annotate(f"T=d={d}", xy=(d, 1), xytext=(d*1.1, 1.5),
                fontsize=7.5, color=color, ha="left")

ax.set_xlabel("Longitud de secuencia $T$ [tokens]")
ax.set_ylabel("FLOPs relativos (norm. en T=d)")
ax.set_title("Crossover $O(T^2 d)$ Attention vs $O(T d^2)$ FFN\n"
             "(sólido: attention, punteado: FFN)", fontweight="bold")

# Leyenda simplificada
from matplotlib.lines import Line2D
handles = [
    Line2D([0],[0], color=c, lw=2, label=l)
    for c,l in zip(colors, labels)
]
handles += [
    Line2D([0],[0], color="k", lw=2, ls="-",  label="Attention: O(T²d)"),
    Line2D([0],[0], color="k", lw=2, ls="--", label="FFN: O(Td²)"),
]
ax.legend(handles=handles, fontsize=8, ncol=2, loc="upper left")
plt.tight_layout()
plt.show()

# ── Regla 6ND ───────────────────────────────────────────────────────────────
print("\n── Regla 6ND: C_train ≈ 6·N·D ─────────────────────────────────────")
models_nd = {
    "GPT-3":        (175e9,  300e9),
    "Chinchilla-70B": (70e9, 1.4e12),
    "LLaMA-2-70B":  (70e9,  2.0e12),
    "LLaMA-3-70B":  (70e9,  15.0e12),
}
print(f"{'Modelo':<20} {'Params':>12} {'Tokens':>12} {'FLOPs (6ND)':>16}")
for mname, (N, D) in models_nd.items():
    flops = 6 * N * D
    print(f"{mname:<20} {N/1e9:>10.0f}B {D/1e12:>10.1f}T {flops:>16.3e}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Desglose de FLOPs por componente para un Transformer típico
# Para un bloque Transformer: d_model, T (sequence length), 4d FFN expansion
def transformer_block_flops(T, d, n_heads=None):
    """Calcula FLOPs de un bloque Transformer (atención + FFN)."""
    # QKV projections: 3 matmuls de (T,d)x(d,d)
    qkv_proj = 3 * 2 * T * d * d
    # Attention scores: Q@K^T  → (T,T)
    attn_scores = 2 * T * T * d
    # Attention weighted sum: A@V
    attn_val = 2 * T * T * d
    # Output projection: (T,d)x(d,d)
    out_proj = 2 * T * d * d
    # FFN: 2 capas, expansión 4x
    ffn = 2 * (2 * T * d * 4*d)
    total = qkv_proj + attn_scores + attn_val + out_proj + ffn
    breakdown = {
        "QKV proj\n$3O(Td^2)$": qkv_proj,
        "Attn scores\n$O(T^2d)$": attn_scores,
        "Attn values\n$O(T^2d)$": attn_val,
        "Out proj\n$O(Td^2)$": out_proj,
        "FFN\n$8O(Td^2)$": ffn,
    }
    return total, breakdown

T, d = 1024, 768  # contexto y d_model similares a GPT-2
total, bd = transformer_block_flops(T, d)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Pie chart del desglose
labels_bd = list(bd.keys())
values_bd = list(bd.values())
colors_bd = ["#1565C0", "#E53935", "#E53935", "#1565C0", "#43A047"]
wedges, texts, autotexts = axes[0].pie(
    values_bd, labels=labels_bd, colors=colors_bd,
    autopct=lambda p: f"{p:.1f}%", startangle=90,
    textprops=dict(fontsize=8),
)
axes[0].set_title(f"Desglose de FLOPs — bloque Transformer\n(T={T}, d={d})", fontweight="bold")

# Evolución del ratio attention/FFN con T
T_range = np.logspace(1, 5, 500)
attn = 2 * T_range**2 * d * 2       # attn scores + values
ffn  = 8 * T_range * d**2           # FFN dominante
ratio = attn / ffn

axes[1].semilogx(T_range, ratio, lw=2.5, color="#E53935")
axes[1].axhline(1, color="k", ls="--", lw=1.5, label="attention = FFN")
axes[1].axvline(d, color="#1565C0", ls=":", lw=1.5, label=f"T = d = {d}")
axes[1].fill_between(T_range, ratio, 1, where=(ratio < 1), alpha=0.15, color="#1565C0",
                     label="FFN domina")
axes[1].fill_between(T_range, ratio, 1, where=(ratio >= 1), alpha=0.15, color="#E53935",
                     label="Attention domina")
axes[1].set_xlabel("Longitud de secuencia $T$")
axes[1].set_ylabel("Ratio FLOPs(Attn) / FLOPs(FFN)")
axes[1].set_title(f"¿Attention o FFN dominan? (d={d})", fontweight="bold")
axes[1].legend(fontsize=9); axes[1].set_ylim(0, 3)
plt.tight_layout()
plt.show()


---
<a id="sec15"></a>
## 15. Resumen, ejercicios y referencias

### Resumen de conceptos clave

| Concepto | Fórmula / Resultado clave | Usado en LLMs |
|----------|--------------------------|---------------|
| Vanishing gradient | $\|{\partial h_T}/{\partial h_0}\| \to 0$ exponencialmente | Motivación del Transformer |
| SVD rank-k | $W_k = U_k\Sigma_k V_k^T$ | LoRA, compresión |
| Backprop manual | $\delta^{(l)} = ((W^{(l+1)})^T \delta^{(l+1)}) \odot f'(\mathbf{z}^{(l)})$ | Entrenamiento |
| Softmax estable | restar $\max(\mathbf{z})$ antes de exp | Logits enormes |
| Temperatura | $\text{softmax}(\mathbf{z}/T)$ | Sampling |
| Top-p nucleus | $\min S : \sum_{t\in S} p_t \geq p$ | Generación de texto |
| Cross-entropy | $-\log p_\theta(x_t \mid x_{<t})$ | Loss de LLMs |
| Entropía | $H = -\sum p_i \log_2 p_i$ | Perplexidad |
| Scaling law Kaplan | $L \sim N^{-\alpha_N}$ | Decisiones de escala |
| Chinchilla | $D_{\text{opt}} \approx 20N$ | Presupuesto de tokens |
| Complejidad attention | $O(T^2 d)$ | Contextos largos |
| Regla 6ND | $C \approx 6ND$ | Estimación de cómputo |

---

### Ejercicios propuestos

**Nivel básico:**
1. Implementar LSTM paso a paso en NumPy y graficar la norma del gradiente vs longitud.
   Comparar con la RNN simple de §2.

**Nivel intermedio:**
2. Extender la red de 2 capas de §7 a 3 capas con BatchNorm. Verificar el gradient check.
3. Implementar un softmax con temperatura y top-p combinados y muestrear 100 secuencias.
   Graficar la distribución de la entropía de cada secuencia.

**Nivel avanzado:**
4. Dado un presupuesto de $C = 10^{22}$ FLOPs, usa la regla Chinchilla para encontrar
   $N_{\text{opt}}$ y $D_{\text{opt}}$. Estima el coste en USD asumiendo $\$2$ por hora
   con una GPU A100 que realiza $312 \times 10^{12}$ FLOPs/s.
5. Implementar Flash Attention simplificado (sin tiling): verificar que la salida coincide
   con la attention estándar usando `torch.allclose`.

---

### Referencias

- **Vaswani et al. (2017)** — Attention is All You Need. [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
- **Kaplan et al. (2020)** — Scaling Laws for Neural Language Models. [arXiv:2001.08361](https://arxiv.org/abs/2001.08361)
- **Hoffmann et al. (2022)** — Training Compute-Optimal LLMs (Chinchilla). [arXiv:2203.15556](https://arxiv.org/abs/2203.15556)
- **Hochreiter & Schmidhuber (1997)** — Long Short-Term Memory. *Neural Computation* 9(8).
- **Karpathy (2022)** — nanoGPT. [github.com/karpathy/nanoGPT](https://github.com/karpathy/nanoGPT)
- **Goodfellow, Bengio, Courville** — Deep Learning (libro). [deeplearningbook.org](https://www.deeplearningbook.org/)
- **Loshchilov & Hutter (2019)** — Decoupled Weight Decay Regularization (AdamW). [arXiv:1711.05101](https://arxiv.org/abs/1711.05101)
